# 08 — Sensitivity to the functional form of the fall-occurrence model

**Manuscript:** *Latent gait instability underlying retrospective fall occurrence in Parkinson’s disease*

## Aim

This notebook tests how strongly the reported probability curve depends on the prespecified probit form. It compares, on **one locked analytic cohort**, three Bayesian models:

1. fixed-slope probit, corresponding at mean covariate values to
   \(P(Y=1\mid s)=\Phi(s-\tau)\);
2. free-slope probit, \(P(Y=1\mid s)=\Phi(\alpha+\beta_s s+\gamma^Tz)\);
3. a parsimonious four-knot restricted cubic spline on the probit scale.

Every model uses the same outcome, participants, age and disease-duration covariates, priors of comparable scale, and the same two-stage propagation of participant-level latent-score uncertainty. The comparison includes PSIS-LOO/ELPD, posterior predictive checks, overlaid probability curves, and \(dP/ds\).

## Interpretive guardrail

The 95% credible interval of \(\tau\) is uncertainty about a **location parameter**. It is not an estimated interval of rapid biological change. The derivative analysis below describes model-implied probability change and cannot establish a biological threshold, clinical cut-point, or causal intervention target.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import warnings
from pathlib import Path

# Keep all library caches inside the writable project area. This matters in
# sandboxed/reproducible executions and has no effect on statistical results.
_launch_cwd = Path.cwd().resolve()
_cache_root = (_launch_cwd.parent if _launch_cwd.name == "notebooks" else _launch_cwd) / ".runtime_cache"
_cache_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(_cache_root))
os.environ.setdefault("MPLCONFIGDIR", str(_cache_root / "matplotlib"))
os.environ.setdefault("PYTENSOR_FLAGS", f"base_compiledir={_cache_root / 'pytensor'}")

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import seaborn as sns
from IPython.display import Markdown, display
from scipy.special import ndtr
from scipy.stats import norm, spearmanr
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

VALIDATION_MODE = os.getenv("LATENT_GAIT_VALIDATION_MODE", "0") == "1"
QUICK_MODE = os.getenv("LATENT_GAIT_QUICK_MODE", "0") == "1"

# Publication defaults. Quick mode is only for code-path validation, never reporting.
DRAWS = int(os.getenv("LATENT_GAIT_DRAWS", "80" if QUICK_MODE else "2000"))
TUNE = int(os.getenv("LATENT_GAIT_TUNE", "80" if QUICK_MODE else "2000"))
CHAINS = int(os.getenv("LATENT_GAIT_CHAINS", "2" if QUICK_MODE else "4"))
CORES = int(os.getenv("LATENT_GAIT_CORES", str(min(CHAINS, 2))))
TARGET_ACCEPT = float(os.getenv("LATENT_GAIT_TARGET_ACCEPT", "0.92" if QUICK_MODE else "0.95"))

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
RESULTS_ROOT = PROJECT_ROOT / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print({
    "python": platform.python_version(),
    "pymc": pm.__version__,
    "arviz": az.__version__,
    "validation_mode": VALIDATION_MODE,
    "quick_mode": QUICK_MODE,
    "draws": DRAWS,
    "tune": TUNE,
    "chains": CHAINS,
})

## Provenance and required inputs

The primary input is the participant-level processed artifact generated by the original notebook 05:

- `data/processed/subject_level_latent_severity_and_falls.csv`.

It contains one row per processed participant with `ID`, `target_bin`, age, disease duration, posterior latent-score mean, and posterior latent-score SD. The same complete-case participant set is locked before fitting every functional-form model.

`LATENT_GAIT_VALIDATION_MODE=1` activates a synthetic schema-compatible fixture solely to test execution. Synthetic results must never be reported in the manuscript.

In [ ]:
OUT_DIR = RESULTS_ROOT / ("_synthetic_validation/08_model_form_processed" if VALIDATION_MODE else "08_model_form_sensitivity_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUBJECT_CANDIDATES = [
    PROJECT_ROOT / "data/processed/subject_level_latent_severity_and_falls.csv",
    PROJECT_ROOT / "data/processed/subject_level_sensitivity_dataset.csv",
    PROJECT_ROOT / "subject_level_latent_severity_and_falls.csv",
]
RAW_CANDIDATES = [PROJECT_ROOT / "data/raw/DatasetIcotMond.xlsx"]
LATENT_CANDIDATES = [
    PROJECT_ROOT / "data/processed/latent_motor_severity_score.csv",
    PROJECT_ROOT / "latent_motor_severity_score.csv",
]

def first_existing(paths):
    return next((p for p in paths if p.exists()), None)

def clean_id(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip()

def synthetic_subject_fixture(n=269, seed=42):
    rg = np.random.default_rng(seed)
    s = rg.normal(size=n)
    age = rg.normal(68, 8, n)
    duration = np.clip(rg.gamma(2.2, 3.0, n), 0.2, 28)
    eta = -0.65 + 0.42*s + 0.12*(age-age.mean())/age.std() + 0.15*(duration-duration.mean())/duration.std()
    y = rg.binomial(1, ndtr(eta))
    return pd.DataFrame({
        "ID": np.arange(1, n+1),
        "Age": age,
        "Duration_years": duration,
        "target_bin": y,
        "latent_severity": s + rg.normal(0, 0.06, n),
        "latent_severity_sd": rg.uniform(0.075, 0.12, n),
    })

def load_subject_data():
    if VALIDATION_MODE:
        return synthetic_subject_fixture(), "synthetic schema-compatible fixture"
    subject_path = first_existing(SUBJECT_CANDIDATES)
    if subject_path is not None:
        return pd.read_csv(subject_path), str(subject_path)
    raw_path = first_existing(RAW_CANDIDATES)
    latent_path = first_existing(LATENT_CANDIDATES)
    if raw_path is None or latent_path is None:
        raise FileNotFoundError("Missing participant-level processed input and reconstruction inputs.")
    raw = pd.read_excel(raw_path)
    raw.columns = raw.columns.str.strip().str.replace("\n", " ", regex=False)
    latent = pd.read_csv(latent_path)
    raw["ID"] = clean_id(raw["ID"])
    latent["ID"] = clean_id(latent["ID"])
    duration_col = "Duration (years)" if "Duration (years)" in raw else "Duration_years"
    clin = (raw.groupby("ID", as_index=False)
            .agg(Age=("Age", "first"), Duration_years=(duration_col, "first"),
                 target_bin=("target_bin", "max")))
    sev = (latent.groupby("ID", as_index=False)
           .agg(latent_severity=("latent_severity", "median"),
                latent_severity_sd=("latent_severity_sd", "median")))
    return clin.merge(sev, on="ID", how="inner", validate="one_to_one"), f"{raw_path} + {latent_path}"

df_subject_raw, DATA_SOURCE = load_subject_data()
print("Data source:", DATA_SOURCE)
display(df_subject_raw.head())

## Lock the analytic cohort and put all quantities on the manuscript scale

The original pipeline standardizes latent severity, age, and disease duration with population SD (`ddof=0`). The score uncertainty is expressed on the original latent scale; after standardization it is divided by the same scale factor. This is a unit conversion, not a change in the uncertainty model.

Rows are removed once, before fitting any model. Thus LOO comparisons cannot be driven by different samples.

In [ ]:
required = ["ID", "Age", "Duration_years", "target_bin", "latent_severity", "latent_severity_sd"]
missing = [c for c in required if c not in df_subject_raw]
assert not missing, f"Missing required columns: {missing}"

df = df_subject_raw[required].copy()
for c in required[1:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

audit = {
    "rows_input": len(df),
    "unique_ID_input": df["ID"].nunique(),
    "duplicate_IDs": int(df["ID"].duplicated().sum()),
    "missing_by_field": df[required].isna().sum().to_dict(),
}
assert audit["duplicate_IDs"] == 0, "Participant-level input must contain one row per ID."
df = df.dropna(subset=required).copy().sort_values("ID").reset_index(drop=True)
assert set(df["target_bin"].unique()).issubset({0, 1}), "target_bin must be binary 0/1."
assert (df["latent_severity_sd"] > 0).all(), "All latent-score SDs must be positive."

sc_sev = StandardScaler().fit(df[["latent_severity"]])
df["sev_z"] = sc_sev.transform(df[["latent_severity"]]).ravel()
df["sev_sd_z"] = df["latent_severity_sd"] / float(sc_sev.scale_[0])
for source, target in [("Age", "age_z"), ("Duration_years", "dur_z")]:
    sc = StandardScaler().fit(df[[source]])
    df[target] = sc.transform(df[[source]]).ravel()

audit.update({
    "rows_locked": len(df),
    "unique_ID_locked": df["ID"].nunique(),
    "events": int(df["target_bin"].sum()),
    "event_fraction": float(df["target_bin"].mean()),
    "latent_scale_factor": float(sc_sev.scale_[0]),
    "mean_sev_sd_z": float(df["sev_sd_z"].mean()),
})
display(pd.Series(audit, name="value").to_frame())

df.to_csv(OUT_DIR / "locked_subject_cohort.csv", index=False)
with open(OUT_DIR / "cohort_audit.json", "w") as f:
    json.dump(audit, f, indent=2)

## Restricted cubic spline specification

The flexible model uses four fixed knots at the 5th, 35th, 65th, and 95th percentiles of observed standardized latent severity. A natural/restricted cubic basis yields one linear and two nonlinear terms (three degrees of freedom for severity). Knots are fixed before observing the outcome. Nonlinear basis columns are standardized using the observed score distribution so the priors remain regularizing and interpretable.

In [ ]:
def rcs_basis_np(x, knots):
    x = np.asarray(x, dtype=float)
    k = np.asarray(knots, dtype=float)
    assert len(k) == 4 and np.all(np.diff(k) > 0)
    cols = []
    for j in range(len(k)-2):
        dj = np.maximum(x-k[j], 0.0)**3
        dkm1 = np.maximum(x-k[-2], 0.0)**3
        dk = np.maximum(x-k[-1], 0.0)**3
        h = dj - dkm1*(k[-1]-k[j])/(k[-1]-k[-2]) + dk*(k[-2]-k[j])/(k[-1]-k[-2])
        cols.append(h)
    return np.column_stack(cols)

def rcs_derivative_np(x, knots):
    x = np.asarray(x, dtype=float)
    k = np.asarray(knots, dtype=float)
    cols = []
    for j in range(len(k)-2):
        dj = 3*np.maximum(x-k[j], 0.0)**2
        dkm1 = 3*np.maximum(x-k[-2], 0.0)**2
        dk = 3*np.maximum(x-k[-1], 0.0)**2
        h = dj - dkm1*(k[-1]-k[j])/(k[-1]-k[-2]) + dk*(k[-2]-k[j])/(k[-1]-k[-2])
        cols.append(h)
    return np.column_stack(cols)

def rcs_basis_pt(x, knots):
    k = np.asarray(knots, dtype=float)
    cols = []
    for j in range(len(k)-2):
        dj = pt.maximum(x-k[j], 0.0)**3
        dkm1 = pt.maximum(x-k[-2], 0.0)**3
        dk = pt.maximum(x-k[-1], 0.0)**3
        h = dj - dkm1*(k[-1]-k[j])/(k[-1]-k[-2]) + dk*(k[-2]-k[j])/(k[-1]-k[-2])
        cols.append(h)
    return pt.stack(cols, axis=1)

KNOTS = np.quantile(df["sev_z"], [0.05, 0.35, 0.65, 0.95])
obs_basis = rcs_basis_np(df["sev_z"].to_numpy(), KNOTS)
BASIS_MEAN = obs_basis.mean(axis=0)
BASIS_SD = obs_basis.std(axis=0, ddof=0)
assert np.all(BASIS_SD > 0)

print("RCS knots:", np.round(KNOTS, 3))

## Fit the three Bayesian models

Priors are weakly regularizing. The record-specific unobserved score is modeled as `Normal(sev_z, sev_sd_z)` in every model. Pointwise log likelihood is retained for PSIS-LOO; posterior predictive fall outcomes are retained for checks.

The fixed-slope model has coefficient 1 for severity. At `age_z = dur_z = 0`, it reduces exactly to the manuscript equation. Covariates are included identically in all three specifications to prevent the functional-form comparison from being confounded by adjustment differences.

In [ ]:
coords = {
    "obs_id": np.arange(len(df)),
    "covariate": ["age_z", "dur_z"],
    "spline_term": ["rcs_1", "rcs_2"],
}
y = df["target_bin"].astype(int).to_numpy()
X_cov = df[["age_z", "dur_z"]].to_numpy()

def sample_model(kind):
    with pm.Model(coords=coords) as model:
        cov = pm.Data("cov", X_cov, dims=("obs_id", "covariate"))
        beta_cov = pm.Normal("beta_cov", 0, 1, dims="covariate")
        true_severity = pm.Normal(
            "true_severity",
            mu=df["sev_z"].to_numpy(),
            sigma=df["sev_sd_z"].to_numpy(),
            dims="obs_id",
        )

        if kind == "fixed_slope_probit":
            tau = pm.Normal("tau", 0, 1.5)
            eta = true_severity - tau + pm.math.dot(cov, beta_cov)
        elif kind == "free_slope_probit":
            intercept = pm.Normal("intercept", 0, 1.5)
            beta_sev = pm.Normal("beta_sev", 0, 1)
            eta = intercept + beta_sev*true_severity + pm.math.dot(cov, beta_cov)
        elif kind == "rcs_probit":
            intercept = pm.Normal("intercept", 0, 1.5)
            beta_linear = pm.Normal("beta_linear", 0, 1)
            beta_nonlinear = pm.Normal("beta_nonlinear", 0, 0.75, dims="spline_term")
            b = (rcs_basis_pt(true_severity, KNOTS) - BASIS_MEAN) / BASIS_SD
            eta = (intercept + beta_linear*true_severity
                   + pm.math.sum(b*beta_nonlinear, axis=1)
                   + pm.math.dot(cov, beta_cov))
        else:
            raise ValueError(kind)

        p_fall = pm.Deterministic("p_fall", pm.math.invprobit(eta), dims="obs_id")
        pm.Bernoulli("falls", p=p_fall, observed=y, dims="obs_id")

        idata = pm.sample(
            draws=DRAWS, tune=TUNE, chains=CHAINS, cores=CORES,
            target_accept=TARGET_ACCEPT, random_seed=RANDOM_SEED,
            return_inferencedata=True,
            idata_kwargs={"log_likelihood": True},
            progressbar=not QUICK_MODE,
        )
        pm.sample_posterior_predictive(
            idata, var_names=["falls", "p_fall"],
            random_seed=RANDOM_SEED, extend_inferencedata=True,
            progressbar=False,
        )
    return model, idata

model_names = ["fixed_slope_probit", "free_slope_probit", "rcs_probit"]
models, idatas = {}, {}
for name in model_names:
    print(f"\nFitting {name}...")
    models[name], idatas[name] = sample_model(name)
    idatas[name].to_netcdf(OUT_DIR / f"{name}.nc")

## Sampling diagnostics

Results are reportable only after checking convergence, effective sample sizes, divergences, and the PSIS Pareto-\(k\) diagnostics below. Quick validation mode intentionally uses too few draws for inferential reporting.

In [ ]:
diag_rows = []
for name, idata in idatas.items():
    var_names = [v for v in ["tau", "intercept", "beta_sev", "beta_linear", "beta_nonlinear", "beta_cov"]
                 if v in idata.posterior]
    sm = az.summary(idata, var_names=var_names, hdi_prob=0.95)
    diag_rows.append({
        "model": name,
        "max_rhat": float(sm["r_hat"].max()),
        "min_ess_bulk": float(sm["ess_bulk"].min()),
        "min_ess_tail": float(sm["ess_tail"].min()),
        "divergences": int(idata.sample_stats["diverging"].sum()),
    })
diagnostics = pd.DataFrame(diag_rows)
display(diagnostics)
diagnostics.to_csv(OUT_DIR / "sampling_diagnostics.csv", index=False)

if not QUICK_MODE:
    if (diagnostics["max_rhat"] > 1.01).any() or (diagnostics["divergences"] > 0).any():
        warnings.warn("At least one model needs sampling review before reporting.")

## PSIS-LOO and ELPD

PSIS-LOO is computed from the same pointwise Bernoulli likelihood for all models. Pairwise ELPD differences use pointwise differences; uncertainty is \(\sqrt{n\,\mathrm{Var}(\Delta_i)}\). A more complex curve is not treated as clearly better unless its ELPD gain exceeds approximately two standard errors and Pareto-\(k\) diagnostics are acceptable.

In [ ]:
loos = {name: az.loo(idata, pointwise=True) for name, idata in idatas.items()}
loo_compare = az.compare(idatas, ic="loo", method="stacking", scale="log")
display(loo_compare)
loo_compare.to_csv(OUT_DIR / "psis_loo_model_comparison.csv")

# Exact refitting for any observation with Pareto k > 0.7. Predictors and
# measurement-error summaries remain observed for the held-out record;
# only that record's fall outcome is left out.
RUN_EXACT_RELOO = os.getenv("LATENT_GAIT_EXACT_RELOO", "1") == "1" and not QUICK_MODE

def _flat_post(idata, var):
    a = idata.posterior[var].stack(sample=("chain", "draw"))
    if a.dims[-1] != "sample":
        a = a.transpose(..., "sample")
    return np.asarray(a)

def exact_loo_logscore(kind, holdout_idx):
    keep = np.arange(len(df)) != holdout_idx
    coords_train = {
        "obs_id": np.arange(keep.sum()),
        "covariate": ["age_z", "dur_z"],
        "spline_term": ["rcs_1", "rcs_2"],
    }
    with pm.Model(coords=coords_train) as model:
        cov = pm.Data("cov", X_cov[keep], dims=("obs_id", "covariate"))
        beta_cov = pm.Normal("beta_cov", 0, 1, dims="covariate")
        true_severity = pm.Normal(
            "true_severity", df.loc[keep, "sev_z"].to_numpy(),
            df.loc[keep, "sev_sd_z"].to_numpy(), dims="obs_id"
        )
        if kind == "fixed_slope_probit":
            tau = pm.Normal("tau", 0, 1.5)
            eta = true_severity - tau + pm.math.dot(cov, beta_cov)
        elif kind == "free_slope_probit":
            intercept = pm.Normal("intercept", 0, 1.5)
            beta_sev = pm.Normal("beta_sev", 0, 1)
            eta = intercept + beta_sev*true_severity + pm.math.dot(cov, beta_cov)
        elif kind == "rcs_probit":
            intercept = pm.Normal("intercept", 0, 1.5)
            beta_linear = pm.Normal("beta_linear", 0, 1)
            beta_nonlinear = pm.Normal("beta_nonlinear", 0, .75, dims="spline_term")
            b = (rcs_basis_pt(true_severity, KNOTS)-BASIS_MEAN)/BASIS_SD
            eta = (intercept + beta_linear*true_severity
                   + pm.math.sum(b*beta_nonlinear, axis=1)
                   + pm.math.dot(cov, beta_cov))
        else:
            raise ValueError(kind)
        pm.Bernoulli("falls", p=pm.math.invprobit(eta), observed=y[keep], dims="obs_id")
        fit = pm.sample(
            draws=DRAWS, tune=TUNE, chains=CHAINS, cores=CORES,
            target_accept=TARGET_ACCEPT, random_seed=RANDOM_SEED+1000+int(holdout_idx),
            return_inferencedata=True, progressbar=not QUICK_MODE,
        )

    n_samples = fit.posterior.sizes["chain"]*fit.posterior.sizes["draw"]
    rg = np.random.default_rng(RANDOM_SEED+2000+int(holdout_idx))
    s = rg.normal(df.loc[holdout_idx, "sev_z"], df.loc[holdout_idx, "sev_sd_z"], n_samples)
    bc = _flat_post(fit, "beta_cov")
    if bc.shape[0] != 2: bc = bc.T
    cov_part = X_cov[holdout_idx] @ bc
    if kind == "fixed_slope_probit":
        eta = s - _flat_post(fit, "tau").reshape(-1) + cov_part
    elif kind == "free_slope_probit":
        eta = (_flat_post(fit, "intercept").reshape(-1)
               + _flat_post(fit, "beta_sev").reshape(-1)*s + cov_part)
    else:
        bn = _flat_post(fit, "beta_nonlinear")
        if bn.shape[0] != 2: bn = bn.T
        b = (rcs_basis_np(s, KNOTS)-BASIS_MEAN)/BASIS_SD
        eta = (_flat_post(fit, "intercept").reshape(-1)
               + _flat_post(fit, "beta_linear").reshape(-1)*s
               + np.sum(b*bn.T, axis=1) + cov_part)
    p = ndtr(eta)
    pred_prob = p.mean() if y[holdout_idx] == 1 else (1-p).mean()
    fit.to_netcdf(OUT_DIR / f"exact_reloo_{kind}_obs_{holdout_idx}.nc")
    return float(np.log(np.clip(pred_prob, 1e-12, 1)))

loo_i_used = {name: np.asarray(loo.loo_i).copy() for name, loo in loos.items()}
reloo_rows = []
if RUN_EXACT_RELOO:
    for name, loo in loos.items():
        bad_indices = np.where(np.asarray(loo.pareto_k) > .7)[0]
        for idx in bad_indices:
            exact = exact_loo_logscore(name, int(idx))
            reloo_rows.append({
                "model": name, "observation_index": int(idx),
                "pareto_k": float(np.asarray(loo.pareto_k)[idx]),
                "psis_loo_i": float(loo_i_used[name][idx]),
                "exact_loo_i": exact,
            })
            loo_i_used[name][idx] = exact
reloo_summary = pd.DataFrame(reloo_rows)
display(reloo_summary)
reloo_summary.to_csv(OUT_DIR / "exact_reloo_corrections.csv", index=False)

loo_rows = []
for name, loo in loos.items():
    k = np.asarray(loo.pareto_k)
    loo_rows.append({
        "model": name,
        "elpd_loo_psis": float(loo.elpd_loo),
        "elpd_loo_after_exact_refits": float(loo_i_used[name].sum()),
        "se": float(loo.se),
        "p_loo": float(loo.p_loo),
        "pareto_k_gt_0.7": int((k > 0.7).sum()),
        "exact_refits": int(sum(r["model"] == name for r in reloo_rows)),
        "pareto_k_max": float(k.max()),
    })
loo_diagnostics = pd.DataFrame(loo_rows)
display(loo_diagnostics)
loo_diagnostics.to_csv(OUT_DIR / "psis_loo_diagnostics.csv", index=False)

pair_rows = []
for alt, ref in [
    ("free_slope_probit", "fixed_slope_probit"),
    ("rcs_probit", "fixed_slope_probit"),
    ("rcs_probit", "free_slope_probit"),
]:
    diff_i = loo_i_used[alt] - loo_i_used[ref]
    pair_rows.append({
        "comparison": f"{alt} minus {ref}",
        "delta_elpd": float(diff_i.sum()),
        "se_delta": float(np.sqrt(len(diff_i)*np.var(diff_i, ddof=1))),
    })
pairwise_elpd = pd.DataFrame(pair_rows)
pairwise_elpd["delta_over_se"] = pairwise_elpd["delta_elpd"] / pairwise_elpd["se_delta"]
display(pairwise_elpd)
pairwise_elpd.to_csv(OUT_DIR / "pairwise_elpd_vs_fixed.csv", index=False)

## Posterior curves and derivative \(dP/ds\)

Curves are evaluated with age and disease duration held at their cohort means. For the spline, the analytic derivative is used. The posterior distribution of the location of maximum derivative is a model-based descriptive summary over the observed range—not a validated threshold.

In [ ]:
def flat(idata, var):
    a = idata.posterior[var].stack(sample=("chain", "draw"))
    if "sample" in a.dims and a.dims[-1] != "sample":
        a = a.transpose(..., "sample")
    return np.asarray(a)

grid = np.linspace(df["sev_z"].min(), df["sev_z"].max(), 301)
curve_samples, deriv_samples = {}, {}

tau = flat(idatas["fixed_slope_probit"], "tau").reshape(-1)
eta = grid[:, None] - tau[None, :]
curve_samples["fixed_slope_probit"] = ndtr(eta)
deriv_samples["fixed_slope_probit"] = norm.pdf(eta)

a = flat(idatas["free_slope_probit"], "intercept").reshape(-1)
b = flat(idatas["free_slope_probit"], "beta_sev").reshape(-1)
eta = a[None, :] + grid[:, None]*b[None, :]
curve_samples["free_slope_probit"] = ndtr(eta)
deriv_samples["free_slope_probit"] = norm.pdf(eta)*b[None, :]

a = flat(idatas["rcs_probit"], "intercept").reshape(-1)
bl = flat(idatas["rcs_probit"], "beta_linear").reshape(-1)
bn = flat(idatas["rcs_probit"], "beta_nonlinear")
if bn.shape[0] != 2:
    bn = bn.T
bg = (rcs_basis_np(grid, KNOTS)-BASIS_MEAN)/BASIS_SD
dbg = rcs_derivative_np(grid, KNOTS)/BASIS_SD
eta = a[None, :] + grid[:, None]*bl[None, :] + bg @ bn
deta = bl[None, :] + dbg @ bn
curve_samples["rcs_probit"] = ndtr(eta)
deriv_samples["rcs_probit"] = norm.pdf(eta)*deta

def summarize_grid(samples, name, quantity):
    return pd.DataFrame({
        "model": name, "quantity": quantity, "sev_z": grid,
        "mean": samples.mean(axis=1),
        "q025": np.quantile(samples, 0.025, axis=1),
        "q975": np.quantile(samples, 0.975, axis=1),
    })

curve_summary = pd.concat([summarize_grid(v, k, "P_fall") for k, v in curve_samples.items()])
deriv_summary = pd.concat([summarize_grid(v, k, "dP_ds") for k, v in deriv_samples.items()])
curve_summary.to_csv(OUT_DIR / "posterior_probability_curves.csv", index=False)
deriv_summary.to_csv(OUT_DIR / "posterior_derivative_curves.csv", index=False)

colors = dict(fixed_slope_probit="#4C72B0", free_slope_probit="#DD8452", rcs_probit="#55A868")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for name in model_names:
    d = curve_summary[curve_summary.model == name]
    axes[0].plot(d.sev_z, d["mean"], color=colors[name], label=name.replace("_", " "))
    axes[0].fill_between(d.sev_z, d.q025, d.q975, color=colors[name], alpha=.12)
    d = deriv_summary[deriv_summary.model == name]
    axes[1].plot(d.sev_z, d["mean"], color=colors[name], label=name.replace("_", " "))
    axes[1].fill_between(d.sev_z, d.q025, d.q975, color=colors[name], alpha=.12)
axes[0].set(xlabel="Latent gait instability (z)", ylabel="Fitted P(fall)", ylim=(0, 1), title="Posterior probability curves")
axes[1].axhline(0, color="0.35", lw=.8)
axes[1].set(xlabel="Latent gait instability (z)", ylabel="dP/ds", title="Model-implied local probability change")
for ax in axes: ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "model_curves_and_derivatives.png", bbox_inches="tight")
plt.show()

max_slope_rows = []
for name, ds in deriv_samples.items():
    loc = grid[np.argmax(ds, axis=0)]
    maxval = np.max(ds, axis=0)
    max_slope_rows.append({
        "model": name,
        "location_mean": float(loc.mean()),
        "location_q025": float(np.quantile(loc, .025)),
        "location_q975": float(np.quantile(loc, .975)),
        "max_derivative_mean": float(maxval.mean()),
        "prob_derivative_positive_everywhere": float((ds.min(axis=0) > 0).mean()),
    })
max_slope = pd.DataFrame(max_slope_rows)
display(max_slope)
max_slope.to_csv(OUT_DIR / "maximum_derivative_location.csv", index=False)

## Posterior predictive checks

Checks cover overall prevalence and calibration across quintiles of latent instability. They assess internal adequacy only and are not external validation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7), sharey=True)
ppc_rows = []
for ax, name in zip(axes, model_names):
    yrep = np.asarray(idatas[name].posterior_predictive["falls"]).reshape(-1, len(df))
    prev = yrep.mean(axis=1)
    ax.hist(prev, bins=25, color=colors[name], alpha=.65)
    ax.axvline(y.mean(), color="black", ls="--", label="observed")
    ax.set(title=name.replace("_", " "), xlabel="Replicated prevalence")
    ppc_rows.append({
        "model": name,
        "observed_prevalence": float(y.mean()),
        "ppc_prevalence_mean": float(prev.mean()),
        "ppc_prevalence_q025": float(np.quantile(prev, .025)),
        "ppc_prevalence_q975": float(np.quantile(prev, .975)),
        "bayesian_p_value_prevalence": float((prev >= y.mean()).mean()),
    })
axes[0].set_ylabel("Posterior predictive draws")
axes[-1].legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "ppc_prevalence.png", bbox_inches="tight")
plt.show()

ppc_summary = pd.DataFrame(ppc_rows)
display(ppc_summary)
ppc_summary.to_csv(OUT_DIR / "ppc_prevalence_summary.csv", index=False)

bins = pd.qcut(df["sev_z"], 5, duplicates="drop")
cal_rows = []
for name in model_names:
    p = np.asarray(idatas[name].posterior["p_fall"]).reshape(-1, len(df)).mean(axis=0)
    tmp = pd.DataFrame({"bin": bins, "y": y, "p": p})
    g = tmp.groupby("bin", observed=True).agg(n=("y", "size"), observed=("y", "mean"), predicted=("p", "mean"))
    g["model"] = name
    cal_rows.append(g.reset_index(drop=True))
calibration = pd.concat(cal_rows, ignore_index=True)
display(calibration)
calibration.to_csv(OUT_DIR / "ppc_calibration_by_severity_quintile.csv", index=False)

## Parameter summaries and reviewer-proof interpretation

The free-slope coefficient quantifies whether the data prefer a sharper or flatter probit curve than the fixed coefficient of 1. The fixed-model \(\tau\) is reported only as the point where the mean-covariate curve has fitted probability 0.5. The spline's derivative summary is treated as exploratory and model dependent.

In [ ]:
parameter_tables = []
for name, vars_ in {
    "fixed_slope_probit": ["tau", "beta_cov"],
    "free_slope_probit": ["intercept", "beta_sev", "beta_cov"],
    "rcs_probit": ["intercept", "beta_linear", "beta_nonlinear", "beta_cov"],
}.items():
    s = az.summary(idatas[name], var_names=vars_, hdi_prob=.95).reset_index().rename(columns={"index":"parameter"})
    s.insert(0, "model", name)
    parameter_tables.append(s)
parameter_summary = pd.concat(parameter_tables, ignore_index=True)
display(parameter_summary)
parameter_summary.to_csv(OUT_DIR / "posterior_parameter_summary.csv", index=False)

fixed_tau = flat(idatas["fixed_slope_probit"], "tau").reshape(-1)
free_beta = flat(idatas["free_slope_probit"], "beta_sev").reshape(-1)
location_summary = pd.DataFrame([{
    "fixed_tau_mean": fixed_tau.mean(),
    "fixed_tau_q025": np.quantile(fixed_tau, .025),
    "fixed_tau_q975": np.quantile(fixed_tau, .975),
    "free_beta_mean": free_beta.mean(),
    "free_beta_q025": np.quantile(free_beta, .025),
    "free_beta_q975": np.quantile(free_beta, .975),
    "P_free_beta_gt_0": (free_beta > 0).mean(),
    "P_free_beta_gt_1": (free_beta > 1).mean(),
}])
display(location_summary)
location_summary.to_csv(OUT_DIR / "location_and_free_slope_summary.csv", index=False)

In [ ]:
def fmt(x): return f"{x:.2f}"

pw = pairwise_elpd.set_index("comparison")
free_row = pw.loc["free_slope_probit minus fixed_slope_probit"]
spline_row = pw.loc["rcs_probit minus fixed_slope_probit"]
nonlinear_row = pw.loc["rcs_probit minus free_slope_probit"]
clear_free = free_row.delta_elpd > 2*free_row.se_delta
clear_spline = spline_row.delta_elpd > 2*spline_row.se_delta
clear_nonlinear = nonlinear_row.delta_elpd > 2*nonlinear_row.se_delta
bad_k = int(loo_diagnostics["pareto_k_gt_0.7"].sum())
reloo_done = bad_k > 0 and len(reloo_rows) == bad_k

if clear_spline:
    form_sentence = ("The flexible spline improved expected out-of-sample fit relative to the fixed-slope probit "
                     f"(ΔELPD={fmt(spline_row.delta_elpd)}, SE={fmt(spline_row.se_delta)}).")
else:
    form_sentence = ("The flexible spline did not show a clear out-of-sample advantage over the fixed-slope probit "
                     f"(ΔELPD={fmt(spline_row.delta_elpd)}, SE={fmt(spline_row.se_delta)}).")
if clear_free:
    slope_sentence = ("Allowing the probit slope to be estimated improved expected predictive fit "
                      f"(ΔELPD={fmt(free_row.delta_elpd)}, SE={fmt(free_row.se_delta)}).")
else:
    slope_sentence = ("Allowing the probit slope to be estimated did not clearly improve expected predictive fit "
                      f"(ΔELPD={fmt(free_row.delta_elpd)}, SE={fmt(free_row.se_delta)}).")

if clear_nonlinear:
    nonlinear_sentence = ("The spline also improved on the free-slope probit, supporting additional nonlinear structure "
                          f"(ΔELPD={fmt(nonlinear_row.delta_elpd)}, SE={fmt(nonlinear_row.se_delta)}).")
else:
    nonlinear_sentence = ("The spline did not improve on the free-slope probit, providing no clear evidence for additional "
                          f"nonlinear structure (ΔELPD={fmt(nonlinear_row.delta_elpd)}, SE={fmt(nonlinear_row.se_delta)}).")

if bad_k == 0:
    warning_sentence = " All PSIS diagnostics were acceptable."
elif reloo_done:
    warning_sentence = (f" PSIS diagnostics flagged {bad_k} observation{'s' if bad_k != 1 else ''} with Pareto k > 0.7; "
                        "the affected pointwise contribution was replaced by an exact leave-one-out refit before computing the reported ELPD differences.")
else:
    warning_sentence = (f" PSIS diagnostics flagged {bad_k} observation{'s' if bad_k != 1 else ''} with Pareto k > 0.7; "
                        "exact or K-fold refitting is required before relying on ELPD differences involving the affected model.")

rebuttal = f"""### Rebuttal-ready result: functional-form sensitivity

Using the same {len(df)} participants, the same age and disease-duration adjustment, and the same propagation of participant-level latent-score uncertainty, we compared the prespecified fixed-slope probit with a free-slope probit and a parsimonious restricted cubic spline. {slope_sentence} The estimated free probit slope was {free_beta.mean():.2f} (95% CrI {np.quantile(free_beta,.025):.2f} to {np.quantile(free_beta,.975):.2f}), compared with the value of 1 imposed by the original formulation. {form_sentence} {nonlinear_sentence}{warning_sentence}

In the fixed-slope model, the posterior mean location parameter was τ={fixed_tau.mean():.2f} (95% CrI {np.quantile(fixed_tau,.025):.2f} to {np.quantile(fixed_tau,.975):.2f}). This interval quantifies uncertainty in the location where the mean-covariate fitted probability equals 0.5; it is not an empirical estimate of a biological transition interval. Across all specifications, any concentration of dP/ds is reported as model-implied local probability change over the observed latent-instability range, not as a clinical threshold or cut-point.

**Decision rule used:** additional nonlinear structure is considered supported only when the spline gains more than approximately two standard errors of ELPD over the free-slope probit and PSIS diagnostics are acceptable. These are internal model comparisons and do not provide external validation.

"""

if VALIDATION_MODE:
    rebuttal = "**SYNTHETIC VALIDATION OUTPUT — DO NOT REPORT.**\n\n" + rebuttal
display(Markdown(rebuttal))
(OUT_DIR / "rebuttal_ready_model_form.md").write_text(rebuttal)

## Reporting checklist

Before copying the generated text into the rebuttal:

- run with the real locked cohort and publication sampling defaults;
- require acceptable convergence and inspect all divergences;
- inspect Pareto-\(k\); use exact LOO or K-fold CV if influential observations invalidate PSIS;
- report ELPD differences with their SE, not model rank alone;
- keep \(\tau\)'s CrI distinct from any interval of steep change;
- describe derivative maxima as exploratory model summaries;
- do not use “safe”, “high risk”, “transition zone”, or “clinical threshold”.